In [ ]:
import pandas as pd
import numpy as np

FILE = "Copy of Master Data_290102026 2 - Copy.xlsx"

df = pd.read_excel(FILE)
df.columns = df.columns.str.strip()

# Clean Category
df["Category"] = (
    df["Category"]
    .astype(str)
    .str.strip()
    .str.lower()
)


In [ ]:
df = df[df["Category"].isin(["repeater", "stranger"])].copy()


In [ ]:
df["effective_daily_demand"] = df["Daily Plan"] * df["Sub Count"]
    

In [ ]:
T120_MACHINES = {"MP-01", "MP-05", "MP-10", "MP-17"}

grp = df.groupby("Child Part", sort=False)

parts = grp.agg({
    "effective_daily_demand": "sum",
    "Inventory_25": "first",
    "Minimum Quantity": "first",
    "Cycle Time": "first",
    "Vertical Machines": lambda x: [
        m.strip()
        for m in ",".join(x.dropna().astype(str)).split(",")
        if m.strip() in T120_MACHINES
    ],
    "Category": "first"
}).reset_index()


In [ ]:
parts = parts[parts["Vertical Machines"].map(len) > 0].copy()


In [ ]:
parts.rename(columns={
    "Child Part": "part",
    "effective_daily_demand": "daily_demand",
    "Inventory_25": "inventory",
    "Minimum Quantity": "min_qty",
    "Cycle Time": "cycle_time",
    "Vertical Machines": "machines"
}, inplace=True)


In [ ]:
parts["net_required_qty"] = (
    parts["daily_demand"]
    + parts["min_qty"]
    - parts["inventory"]
).clip(lower=0)


In [ ]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m,
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("No Repeater/Stranger parts found for 120T machines")


In [ ]:
rows = []

for _, r in parts.iterrows():
    for m in r["machines"]:
        rows.append({
            "part": r["part"],
            "category": r["Category"],
            "machine": m,
            "daily_demand": r["daily_demand"],
            "inventory": r["inventory"],
            "cycle_time": r["cycle_time"],
            "net_required_qty": r["net_required_qty"]
        })

dfm = pd.DataFrame(rows)

if dfm.empty:
    raise ValueError("No Repeater/Stranger parts found for 120T machines")


In [ ]:
CHANGEOVER = 40
CAPACITY = 1320
TARGET_DAYS = 3

def compute_score(row):
    inv_days = (
        row["inventory"] / row["daily_demand"]
        if row["daily_demand"] > 0 else TARGET_DAYS
    )
    inventory_pain = max(0, TARGET_DAYS - inv_days)

    target_qty = TARGET_DAYS * row["daily_demand"]
    qty_if_made = min(row["net_required_qty"], target_qty)

    relief_days = (
        (qty_if_made + row["inventory"]) / row["daily_demand"]
        if row["daily_demand"] > 0 else 0
    )
    relief = min(relief_days, TARGET_DAYS)

    prod_time = qty_if_made * row["cycle_time"]

    setup_eff = prod_time / CHANGEOVER if prod_time > 0 else 0
    setup_penalty = 1 / (1 + setup_eff)

    machine_share = prod_time / CAPACITY
    monopoly_penalty = max(0, machine_share - 0.6)

    score = (
        3.0 * inventory_pain +
        2.0 * relief -
        2.5 * setup_penalty -
        3.0 * monopoly_penalty
    )

    return round(score, 3)


In [ ]:
dfm["score"] = dfm.apply(compute_score, axis=1)


In [ ]:
selected = []

for m, g in dfm.groupby("machine"):
    g = g.sort_values("score", ascending=False)
    selected.append(g.head(3))

selected = pd.concat(selected).reset_index(drop=True)


In [ ]:
def qty_today(row):
    target_qty = TARGET_DAYS * row["daily_demand"]
    qty = max(0, target_qty - row["inventory"])
    return int(min(qty, row["net_required_qty"]))

selected["qty_today"] = selected.apply(qty_today, axis=1)
selected["prod_time_min"] = selected["qty_today"] * selected["cycle_time"]


In [ ]:
final_plan = selected[[
    "machine",
    "part",
    "category",
    "qty_today",
    "prod_time_min",
    "score"
]].sort_values(["machine", "score"], ascending=[True, False])

print("\n===== DAILY PLAN (120T MACHINES ONLY) =====")
print(final_plan)
